
# **CycleGAN Evaluation Script**

### Folder structure expected:

```
 dataset/
   ├── monet_jpg/   (REAL Monet)          = domain A real
   ├── photo_jpg/   (REAL Photo)          = domain B real
   ├── pred_A2B/    (GEN Monet -> Photo)  = generated B
   └── pred_B2A/    (GEN Photo -> Monet)  = generated A
```

### The below script computes the average in both directions of:
- `FID_A2B, MiFID_A2B`:  real photos  vs generated photos (pred_A2B)
- `FID_B2A, MiFID_B2A`:  real monet   vs generated monet  (pred_B2A)



In [ ]:
import os
print(os.listdir())
print(os.listdir("dataset"))

['.ipynb_checkpoints', 'checkpoints', 'cycle_gan.ipynb', 'dataset', 'loss_curves.png', 'loss_history.csv', 'Part3_Evaluation_Script.ipynb', 'visual_check.png']
['monet_jpg', 'photo_jpg', 'pred_A2B', 'pred_B2A']


In [ ]:
!pip -q install scipy pillow tqdm

In [ ]:
import os, glob
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models

import scipy.linalg
from scipy.spatial.distance import cosine

In [ ]:
# Config: set your root
BASE = "dataset"

REAL_MONET = os.path.join(BASE, "monet_jpg")   # real domain A
REAL_PHOTO = os.path.join(BASE, "photo_jpg")   # real domain B
GEN_A2B    = os.path.join(BASE, "pred_A2B")    # Monet -> Photo #You can modify the patha s per your directory
GEN_B2A    = os.path.join(BASE, "pred_B2A")    # Photo -> Monet #You can modify the patha s per your directory

# Limit to N images per set
# Set to None to use all images in each folder
N_EVAL = 300

BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
def list_images(folder):
    exts = (".jpg", ".jpeg", ".png")
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(folder, f"*{ext}")))
        paths.extend(glob.glob(os.path.join(folder, f"*{ext.upper()}")))
    paths = sorted(list(set(paths)))
    return paths

def take_n(paths, n):
    if n is None:
        return paths
    return paths[:min(n, len(paths))]

In [ ]:
# Inception feature extractor
def get_inception_model():
    inception = models.inception_v3(
        weights=models.Inception_V3_Weights.IMAGENET1K_V1,
        transform_input=False
    )
    inception.fc = nn.Identity()
    inception.to(device)
    inception.eval()
    return inception

INCEPTION_TF = T.Compose([
    T.Resize(299),
    T.CenterCrop(299),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

def load_batch(paths):
    imgs = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        imgs.append(INCEPTION_TF(img))
    return torch.stack(imgs, dim=0)

@torch.no_grad()
def get_activations(model, image_paths, batch_size=32):
    feats = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Inception activations"):
        batch_paths = image_paths[i:i+batch_size]
        x = load_batch(batch_paths).to(device)
        f = model(x).detach().cpu().numpy()
        feats.append(f)
    return np.concatenate(feats, axis=0)

In [ ]:
def frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    covmean, _ = scipy.linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = scipy.linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    diff = mu1 - mu2
    return float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean))

In [ ]:
def calculate_fid_mifid(real_paths, gen_paths, batch_size=32, subsample_to_match=True):
    # For fair comparison, match counts
    real_paths = sorted(real_paths)
    gen_paths  = sorted(gen_paths)

    if subsample_to_match:
        n = min(len(real_paths), len(gen_paths))
        real_paths = real_paths[:n]
        gen_paths  = gen_paths[:n]

    model = get_inception_model()

    real_act = get_activations(model, real_paths, batch_size=batch_size)
    gen_act  = get_activations(model, gen_paths,  batch_size=batch_size)

    mu_r, sig_r = real_act.mean(axis=0), np.cov(real_act, rowvar=False)
    mu_g, sig_g = gen_act.mean(axis=0),  np.cov(gen_act,  rowvar=False)

    fid = frechet_distance(mu_r, sig_r, mu_g, sig_g)

    # mean cosine distance between feature vectors,
    # paired by index after subsampling/matching
    m = min(len(real_act), len(gen_act))
    cos_dists = [cosine(real_act[i], gen_act[i]) for i in range(m)]
    mifid = float(np.mean(cos_dists))

    return fid, mifid

In [ ]:
for d in [REAL_MONET, REAL_PHOTO, GEN_A2B, GEN_B2A]:
    assert os.path.isdir(d), f"Missing folder: {d}"

real_monet = take_n(list_images(REAL_MONET), N_EVAL)
real_photo = take_n(list_images(REAL_PHOTO), N_EVAL)
gen_a2b    = take_n(list_images(GEN_A2B),    N_EVAL)  # Monet->Photo (generated photos)
gen_b2a    = take_n(list_images(GEN_B2A),    N_EVAL)  # Photo->Monet (generated monet)

print("\nCounts (after N_EVAL cap):")
print("Real Monet:", len(real_monet), " | Gen Monet (B2A):", len(gen_b2a))
print("Real Photo:", len(real_photo), " | Gen Photo (A2B):", len(gen_a2b))


Counts (after N_EVAL cap):
Real Monet: 300  | Gen Monet (B2A): 300
Real Photo: 300  | Gen Photo (A2B): 300


In [ ]:
print("\n Evaluating Photo -> Monet (B2A) ")
# Ground truth = real Monet; Generated = pred_B2A (generated Monet-like)
fid_B2A, mifid_B2A = calculate_fid_mifid(real_monet, gen_b2a, batch_size=BATCH_SIZE)
print(f"[Photo->Monet] FID={fid_B2A:.3f}  MiFID={mifid_B2A:.4f}")

print("\n Evaluating Monet -> Photo (A2B) ")
# Ground truth = real Photo; Generated = pred_A2B (generated Photo-like)
fid_A2B, mifid_A2B = calculate_fid_mifid(real_photo, gen_a2b, batch_size=BATCH_SIZE)
print(f"[Monet->Photo] FID={fid_A2B:.3f}  MiFID={mifid_A2B:.4f}")



 Evaluating Photo -> Monet (B2A) 
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:07<00:00, 15.4MB/s] 
Inception activations: 100%|██████████| 10/10 [00:03<00:00,  2.97it/s]
/tmp/ipykernel_43822/2390798440.py:2: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = scipy.linalg.sqrtm(sigma1.dot(sigma2), disp=False)


[Photo->Monet] FID=95.269  MiFID=0.4041

 Evaluating Monet -> Photo (A2B) 


Inception activations: 100%|██████████| 10/10 [00:03<00:00,  3.25it/s]


[Monet->Photo] FID=96.582  MiFID=0.4137


In [ ]:
import pandas as pd

sub_fid  = (fid_A2B + fid_B2A) / 2
sub_mifid = (mifid_A2B + mifid_B2A) / 2

submission = pd.DataFrame([{
    "ID": 1,
    "FID": float(sub_fid),
    "MiFID": float(sub_mifid)}])

submission.to_csv("submission.csv", index=False)
print(submission)

   ID        FID   MiFID
0   1  95.925299  0.4089
